# Transformer Encoder for NLE ML Replacement (Iteration 3)
This notebook trains a Transformer Encoder on the full >500k dataset to replace a conventional DSP Nonlinear Equalizer (NLE).
We evaluate whether self-attention and sequence context provide a meaningful improvement for nonlinear equalization compared with a memoryless MLP.


## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import copy
import math
import os
import pandas as pd
import gc


## 2. Configuration & Hyperparameters

In [ ]:
# Model Config V3
WINDOW = 256
STRIDE = 128
D_MODEL = 128
NHEAD = 8
NUM_LAYERS = 4
DIM_FEEDFORWARD = 512
DROPOUT = 0.1
USE_POSITIONAL_ENCODING = True

# Training Config
TARGET_BATCH_SIZE = 64
EPOCHS = 200
LR = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 25

# Loss Weights V3
OUTPUT_WEIGHT = 0.5
AMP_WEIGHT = 0.5
SNAP_WEIGHT = 0.005
DC_WEIGHT = 0.1
EPSILON = 1e-8


## 3. Device Detection & Reproducibility

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training device:", device)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


## 4. Load Full Dataset
We load the pre-extracted full dataset from `data/nle_off.npy` and `data/nle_on.npy`.

In [ ]:
off_data = np.load('data/nle_off.npy')
on_data = np.load('data/nle_on.npy')

assert len(off_data) == len(on_data), "Mismatch in dataset lengths!"
print(f"Loaded full dataset. Total samples: {len(off_data)}")


## 5. Dataset Statistics & Joint Normalization

In [ ]:
def print_stats(name, data):
    rms = np.sqrt(np.mean(data**2))
    print(f"--- {name} Stats ---")
    print(f"Shape: {data.shape}")
    print(f"Min: {np.min(data, axis=0)}")
    print(f"Max: {np.max(data, axis=0)}")
    print(f"Mean: {np.mean(data, axis=0)}")
    print(f"Std: {np.std(data, axis=0)}")
    print(f"RMS Magnitude: {rms:.4f}
")

print_stats("NLE OFF", off_data)
print_stats("DSP NLE ON", on_data)

mean_val = np.mean(off_data, axis=0)
std_val = np.std(off_data, axis=0)

off_data_norm = (off_data - mean_val) / std_val
on_data_norm = (on_data - mean_val) / std_val

print("After Joint Normalization:")
print_stats("NLE OFF (Norm)", off_data_norm)
print_stats("DSP NLE ON (Norm)", on_data_norm)


## 6. Train / Validation / Test Split (70/15/15)
Split the continuous signal BEFORE windowing to prevent leakage.

In [ ]:
total_samples = len(off_data_norm)
train_end = int(total_samples * 0.70)
val_end = int(total_samples * 0.85)

train_off = off_data_norm[:train_end]
train_on = on_data_norm[:train_end]

val_off = off_data_norm[train_end:val_end]
val_on = on_data_norm[train_end:val_end]

test_off = off_data_norm[val_end:]
test_on = on_data_norm[val_end:]

print(f"Train split size: {len(train_off)} ({len(train_off)/total_samples*100:.1f}%)")
print(f"Val split size:   {len(val_off)} ({len(val_off)/total_samples*100:.1f}%)")
print(f"Test split size:  {len(test_off)} ({len(test_off)/total_samples*100:.1f}%)")


## 7. Sequence Windowing
Generate overlapping sequence windows of size 256 for each split independently.

In [ ]:
class NLEWindowDataset(Dataset):
    def __init__(self, x_data, y_data, window, stride):
        self.x_windows = []
        self.delta_windows = []
        self.y_windows = []
        
        delta_data = y_data - x_data
        num_windows = (len(x_data) - window) // stride + 1
        for i in range(num_windows):
            start = i * stride
            end = start + window
            self.x_windows.append(x_data[start:end])
            self.delta_windows.append(delta_data[start:end])
            self.y_windows.append(y_data[start:end])
            
    def __len__(self):
        return len(self.x_windows)
        
    def __getitem__(self, idx):
        return torch.tensor(self.x_windows[idx], dtype=torch.float32),                torch.tensor(self.delta_windows[idx], dtype=torch.float32),                torch.tensor(self.y_windows[idx], dtype=torch.float32)

train_dataset = NLEWindowDataset(train_off, train_on, WINDOW, STRIDE)
val_dataset = NLEWindowDataset(val_off, val_on, WINDOW, STRIDE)
test_dataset = NLEWindowDataset(test_off, test_on, WINDOW, STRIDE)

print(f"Train windows: {len(train_dataset)}")
print(f"Val windows:   {len(val_dataset)}")
print(f"Test windows:  {len(test_dataset)}")


## 8. Transformer Model V3
Expanded capacity and pre-normalization.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerNLE_v3(nn.Module):
    def __init__(self, d_model, nhead, num_layers, dim_feedforward, dropout, use_pe):
        super().__init__()
        self.use_pe = use_pe
        self.input_linear = nn.Linear(2, d_model)
        if self.use_pe:
            self.pos_encoder = PositionalEncoding(d_model)
            
        try:
            encoder_layers = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, 
                dropout=dropout, batch_first=True, norm_first=True
            )
        except TypeError:
            encoder_layers = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, 
                dropout=dropout, batch_first=True
            )
            
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.output_linear = nn.Linear(d_model, 2)
        
    def forward(self, src):
        x = self.input_linear(src)
        if self.use_pe:
            x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        out = self.output_linear(x)
        return out

model = TransformerNLE_v3(D_MODEL, NHEAD, NUM_LAYERS, DIM_FEEDFORWARD, DROPOUT, USE_POSITIONAL_ENCODING).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Transformer V3 Parameter Count: {count_parameters(model):,}")


## 9. Batch Size Memory Check
Test memory limit on GPU and gracefully degrade batch size if OOM occurs.

In [ ]:
def find_safe_batch_size(target_bs):
    if device.type == 'cpu': return target_bs
    bs = target_bs
    while bs >= 16:
        try:
            dummy_x = torch.randn(bs, WINDOW, 2, device=device)
            out = model(dummy_x)
            out.sum().backward()
            model.zero_grad()
            torch.cuda.empty_cache()
            return bs
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"OOM at batch size {bs}. Trying {bs//2}...")
                bs //= 2
                model.zero_grad()
                torch.cuda.empty_cache()
            else:
                raise e
    return bs

safe_bs = find_safe_batch_size(TARGET_BATCH_SIZE)
print(f"Selected Batch Size: {safe_bs}")

train_loader = DataLoader(train_dataset, batch_size=safe_bs, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=safe_bs, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=safe_bs, shuffle=False)


## 10. V3 Domain-Aware Loss
Includes the direct DSP NLE ON output MSE as an explicit objective.

In [ ]:
def domain_aware_loss_v3(pred_delta, true_delta, x, y_true):
    # 1. IQ Correction MSE (Delta)
    mse_delta = torch.mean((pred_delta - true_delta)**2)
    
    y_hat = x + pred_delta
    
    # 2. Direct Output MSE
    mse_output = torch.mean((y_hat - y_true)**2)
    
    # 3. Amplitude Preservation
    amp_hat = torch.sqrt(y_hat[..., 0]**2 + y_hat[..., 1]**2 + EPSILON)
    amp_true = torch.sqrt(y_true[..., 0]**2 + y_true[..., 1]**2 + EPSILON)
    mse_amp = torch.mean((amp_hat - amp_true)**2)
    
    # 4. Dynamic QPSK Snapping
    dynamic_radius = torch.mean(amp_true)
    ideal_mag = dynamic_radius.item() / math.sqrt(2)
    
    ideal_points = torch.tensor([
        [-ideal_mag, -ideal_mag], [-ideal_mag, ideal_mag],
        [ideal_mag, -ideal_mag], [ideal_mag, ideal_mag]
    ], device=x.device)
    
    distances = torch.sum((y_hat.unsqueeze(2) - ideal_points.view(1, 1, 4, 2))**2, dim=-1)
    min_distances, _ = torch.min(distances, dim=-1)
    snap_loss = torch.mean(min_distances)
    
    # 5. DC Centering
    mean_I = torch.mean(y_hat[..., 0])
    mean_Q = torch.mean(y_hat[..., 1])
    dc_loss = mean_I**2 + mean_Q**2
    
    total_loss = mse_delta + OUTPUT_WEIGHT * mse_output + AMP_WEIGHT * mse_amp + SNAP_WEIGHT * snap_loss + DC_WEIGHT * dc_loss
    
    return total_loss, mse_delta.item(), mse_output.item(), mse_amp.item(), snap_loss.item(), dc_loss.item()


## 11. Optimizer and Scheduler

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


## 12. Training Loop with AMP
Leveraging PyTorch Mixed Precision (AMP) if running on CUDA.

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_mse_delta': [], 'train_mse_out': [], 'train_mse_amp': [], 'train_snap': [], 'train_dc': []
}

best_val_loss = float('inf')
patience_counter = 0

use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda') if use_amp else None

for epoch in range(EPOCHS):
    model.train()
    epoch_metrics = np.zeros(6) # total, delta, output, amp, snap, dc
    
    for x_batch, delta_batch, y_batch in train_loader:
        x_batch, delta_batch, y_batch = x_batch.to(device), delta_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            with torch.amp.autocast('cuda'):
                pred_delta = model(x_batch)
                total_loss, mse_delta, mse_out, mse_amp, snap, dc = domain_aware_loss_v3(pred_delta, delta_batch, x_batch, y_batch)
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            pred_delta = model(x_batch)
            total_loss, mse_delta, mse_out, mse_amp, snap, dc = domain_aware_loss_v3(pred_delta, delta_batch, x_batch, y_batch)
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
        epoch_metrics += np.array([total_loss.item(), mse_delta, mse_out, mse_amp, snap, dc])
        
    scheduler.step()
    epoch_metrics /= len(train_loader)
    
    history['train_loss'].append(epoch_metrics[0])
    history['train_mse_delta'].append(epoch_metrics[1])
    history['train_mse_out'].append(epoch_metrics[2])
    history['train_mse_amp'].append(epoch_metrics[3])
    history['train_snap'].append(epoch_metrics[4])
    history['train_dc'].append(epoch_metrics[5])
    
    # Validation
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for x_batch, delta_batch, y_batch in val_loader:
            x_batch, delta_batch, y_batch = x_batch.to(device), delta_batch.to(device), y_batch.to(device)
            if use_amp:
                with torch.amp.autocast('cuda'):
                    pred_delta = model(x_batch)
                    loss_val, _, _, _, _, _ = domain_aware_loss_v3(pred_delta, delta_batch, x_batch, y_batch)
            else:
                pred_delta = model(x_batch)
                loss_val, _, _, _, _, _ = domain_aware_loss_v3(pred_delta, delta_batch, x_batch, y_batch)
            epoch_val_loss += loss_val.item()
            
    epoch_val_loss /= len(val_loader)
    history['val_loss'].append(epoch_val_loss)
    
    print(f"Epoch {epoch+1:03d} | LR: {scheduler.get_last_lr()[0]:.2e} | Train Loss: {epoch_metrics[0]:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    # Early Stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), 'transformer_nle_v3_best.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping triggered after {epoch+1} epochs!")
            break


## 13. Loss Component Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Total Train/Val
axes[0].plot(history['train_loss'], label='Train Total')
axes[0].plot(history['val_loss'], label='Val Total')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# Components
axes[1].plot(history['train_mse_delta'], label='Delta MSE (wt=1.0)')
axes[1].plot(np.array(history['train_mse_out']) * OUTPUT_WEIGHT, label=f'Output MSE (wt={OUTPUT_WEIGHT})')
axes[1].plot(np.array(history['train_mse_amp']) * AMP_WEIGHT, label=f'Amp Loss (wt={AMP_WEIGHT})')
axes[1].plot(np.array(history['train_snap']) * SNAP_WEIGHT, label=f'Snap Loss (wt={SNAP_WEIGHT})')
axes[1].plot(np.array(history['train_dc']) * DC_WEIGHT, label=f'DC Loss (wt={DC_WEIGHT})')
axes[1].set_title('Training Loss Components')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.show()


## 14. Unseen Test-Set Inference & Overlap Reconstruction
Evaluate strictly on the isolated 15% Test Split.

In [ ]:
model.load_state_dict(torch.load('transformer_nle_v3_best.pth', map_location=device, weights_only=True))
model.eval()

accumulated_preds = np.zeros_like(test_off)
overlap_counts = np.zeros(len(test_off))

idx = 0
with torch.no_grad():
    for x_batch, _, _ in test_loader:
        x_batch = x_batch.to(device)
        if use_amp:
            with torch.amp.autocast('cuda'):
                pred_delta_batch = model(x_batch).cpu().numpy()
        else:
            pred_delta_batch = model(x_batch).cpu().numpy()
            
        for i in range(len(pred_delta_batch)):
            start = (idx + i) * STRIDE
            end = start + WINDOW
            
            accumulated_preds[start:end] += pred_delta_batch[i]
            overlap_counts[start:end] += 1
            
        idx += len(pred_delta_batch)

overlap_counts[overlap_counts == 0] = 1
final_delta_hat = accumulated_preds / overlap_counts[:, None]
final_y_hat_norm = test_off + final_delta_hat

# Inverse transform normalization for Test Set
test_off_raw = test_off * std_val + mean_val
test_on_raw = test_on * std_val + mean_val
final_y_hat = final_y_hat_norm * std_val + mean_val

print("Original Test Input Shape:", test_off_raw.shape)
print("Reconstructed Output Shape:", final_y_hat.shape)
assert test_off_raw.shape == final_y_hat.shape, "Shape mismatch after reconstruction!"


## 15. Constellation Plots (Test Set)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot subset for clarity
subset = 10000 if len(test_off_raw) > 10000 else len(test_off_raw)

def plot_constellation(ax, data, title):
    ax.scatter(data[:subset, 0], data[:subset, 1], s=1, alpha=0.5, color='b')
    ax.set_title(title)
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.grid(True)
    ax.set_aspect('equal', 'box')
    ax.set_xlim([-3, 3])
    ax.set_ylim([-3, 3])
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)

plot_constellation(axes[0], test_off_raw, "DSP NLE OFF")
plot_constellation(axes[1], final_y_hat, "Transformer ML NLE V3")
plot_constellation(axes[2], test_on_raw, "DSP NLE ON (Reference)")

plt.tight_layout()
plt.show()


## 16. EVM & MSE-to-DSP Analysis

In [ ]:
def calculate_evm(signal, reference):
    error = signal - reference
    evm_rms = np.sqrt(np.mean(np.sum(error**2, axis=1))) / np.sqrt(np.mean(np.sum(reference**2, axis=1)))
    return evm_rms * 100

evm_off = calculate_evm(test_off_raw, test_on_raw)
evm_ml = calculate_evm(final_y_hat, test_on_raw)

mse_off_vs_on = np.mean((test_off_raw - test_on_raw)**2)
mse_ml_vs_on = np.mean((final_y_hat - test_on_raw)**2)

norm_improvement = (1 - (mse_ml_vs_on / mse_off_vs_on)) * 100

print(f"EVM (DSP NLE OFF): {evm_off:.2f}%")
print(f"EVM (Transformer V3): {evm_ml:.2f}%")
print(f"MSE (NLE OFF -> DSP NLE ON): {mse_off_vs_on:.6f}")
print(f"MSE (Transformer -> DSP NLE ON): {mse_ml_vs_on:.6f}")
print(f"Normalized Improvement: {norm_improvement:.2f}% (Higher is better)")


## 17. BER Calculation
Sweeping phase shifts to correctly report the best aligned BER mapping.

In [ ]:
def qpsk_demodulate_all_phases(signal):
    bits = np.zeros((len(signal), 2), dtype=int)
    bits[:, 0] = (signal[:, 0] > 0).astype(int)
    bits[:, 1] = (signal[:, 1] > 0).astype(int)
    return bits

bits_ref = qpsk_demodulate_all_phases(test_on_raw)

def get_best_ber(signal):
    best_ber = 1.0
    for phase in [0, np.pi/2, np.pi, 3*np.pi/2]:
        c, s = np.cos(phase), np.sin(phase)
        rot_matrix = np.array([[c, -s], [s, c]])
        rotated = signal @ rot_matrix.T
        bits_rot = qpsk_demodulate_all_phases(rotated)
        ber = np.mean(bits_rot != bits_ref)
        if ber < best_ber:
            best_ber = ber
    return best_ber

ber_off = get_best_ber(test_off_raw)
ber_ml = get_best_ber(final_y_hat)

print(f"Minimum BER (DSP NLE OFF): {ber_off:.6f}")
print(f"Minimum BER (Transformer V3): {ber_ml:.6f}")


## 18. Model Comparison and Conclusion

| Model | BER | EVM (%) | MSE to DSP NLE ON |
|---|---|---|---|
| NLE OFF (Distorted) | **{ber_off}** | **{evm_off}** | **{mse_off}** |
| MLP (Baseline) | *Fill from previous* | *Fill from previous* | *Fill from previous* |
| Transformer V3 | **{ber_ml}** | **{evm_ml}** | **{mse_ml}** |
| DSP NLE ON (Teacher) | 0.00 | 0.00 | 0.00 |

### Scientific Conclusion
- **What improved compared to V1/V2?**: *Observe the tightness of the clusters and convergence of the new Direct Output Loss.*
- **Did Transformer V3 beat the MLP?**: *Compare BER and MSE_to_DSP metrics against the MLP baseline using identical test procedures.*
- **How close is it to DSP NLE ON?**: *Refer to the Normalized Improvement metric. A value near 100% means perfect replication.*
- **Is the computational complexity justified?**: *The Transformer V3 contains ~1 Million parameters. If the MLP was significantly smaller but achieved comparable MSE/BER, the sequence attention context may not justify the computational penalty.*
